# HMM Regime Detection Signal

This notebook asks a specific research question:

> Can we infer market risk regimes from observable market, factor, macro, and volatility data, and can those regimes reduce downside risk when used as an exposure overlay on a momentum strategy?

In plain English:

Can we look at data we would have known at the time, decide whether the market looks calm, mixed, or stressed, and use that second opinion to decide how much of a momentum strategy we should hold?

Momentum is still the trading strategy. The regime model is not picking stocks. It is a risk-control layer placed on top of momentum.

Example:

```text
Momentum strategy says: be 100% invested in the long/short momentum portfolio.

Regime model says: the market looks risk-on.
Overlay allows: close to 100% of the momentum exposure.

Regime model says: the market looks risk-off.
Overlay allows: much less of the momentum exposure.
```

## What Exactly Are We Observing?

The phrase "market returns" is easy to misunderstand, so this notebook uses one explicit definition:

- `mkt_returns` is the equal-weighted daily arithmetic return across the cleaned adjusted-close stock price panel: `prices.pct_change(fill_method=None).mean(axis=1)`.
- It is not a sector return, cap-weighted index return, log return, or individual stock return.
- It is daily, not a rolling cumulative return. When the notebook plots market equity or stress-window returns, it compounds these daily arithmetic returns with `(1 + return).cumprod()` or `(1 + return).prod() - 1`.
- The HMM does not directly trade this market return. It uses it only to name otherwise anonymous HMM states: the state with the highest trailing in-sample average `mkt_returns` is called `risk_on`, and the lowest is called `risk_off`.

The HMM observes a feature matrix built from:

- cross-sectional return dispersion from adjusted-close returns;
- cross-sectional beta spread and median volatility from the factor panel;
- lagged macro z-score features;
- VIX as an option-implied stress proxy.

## What Are Regimes?

A regime is a recurring market environment. Here the labels mean:

- `risk_on`: the model's learned state with the strongest trailing market-return behavior. It should generally look calmer or more supportive for risk-taking.
- `neutral`: the middle state. It is a transition or mixed environment, not a bullish or bearish claim by itself.
- `risk_off`: the model's learned state with the weakest trailing market-return behavior. It should generally line up with stress: higher VIX, higher dispersion, higher volatility, weaker returns, or drawdowns.

These are inferred labels, not truth labels. The notebook has to earn them with diagnostics. If `risk_off` does not actually show worse returns, higher realized volatility, or known stress clustering, then the label is not economically credible.

## What Does "Today Looks High-Risk" Mean?

It means today's observed feature pattern, after walk-forward standardization using only trailing data, is closest to the learned `risk_off` state. More specifically:

- each monthly fit uses the trailing 5 years of feature history;
- scaling is fit only on that trailing window, then applied to the next prediction block;
- probabilities are filtered, so a date's probability uses observations only through that date;
- the state label mapping is held fixed inside the 21-day prediction block;
- a high `p_risk_off` means "observations through today resemble historical risk-off states from the training window."

## What Is The Overlay?

An overlay is a second layer placed on top of an existing strategy. It does not replace the momentum signal. It only changes how much exposure we allow.

Here the exposure rule is disclosed explicitly:

```text
raw momentum return on day t = unscaled_mom[t]
regime exposure allowed on day t = yesterday's clipped p_risk_on, from 0.0 to 1.0
regime-scaled momentum return on day t = unscaled_mom[t] * exposure[t - 1]
```

So if yesterday's `p_risk_on` is `0.80`, the notebook applies about 80% of the normal momentum exposure today. If it is `0.20`, the notebook applies about 20%. The one-day lag is intentional: it avoids using today's close information to size today's return.

## What We Want To Discover

1. **EDA:** What does each input variable look like, what transformation created it, and does it visibly react during known stress periods?
2. **Regime inference:** Does a walk-forward HMM find persistent states that line up with interpretable market environments?
3. **Transparency:** When the model says a day is high-risk, can we point to the variables and diagnostics that made that label plausible?
4. **Trading relevance:** Does scaling `mom_12_1` exposure by lagged `p_risk_on` improve Sortino, max drawdown, Calmar, CVaR, or loss probability versus unscaled momentum?
5. **Decision:** If the overlay only lowers volatility by mechanically shrinking exposure, but does not improve drawdown or downside-adjusted performance, it is not yet a useful strategy signal.

## Pipeline

1. Load data from parquet and fetch VIX.
2. Build the feature matrix and document each variable's construction.
3. Plot each variable over time and inspect stress-window behavior.
4. Run walk-forward HMM and plot regime probabilities with market context.
5. Convert lagged `p_risk_on` into an exposure scale from 0% to 100%.
6. Backtest `mom_12_1` with and without that exposure overlay.
7. Interpret whether the overlay improves downside risk, and whether it beats simpler transparent baselines.

**References:**
- Hamilton (1989), *A New Approach to the Economic Analysis of Nonstationary Time Series*
- Ang & Bekaert (2002), *International Asset Allocation with Regime Shifts*
- Daniel and Moskowitz (2016), *Momentum Crashes*

In [ ]:
import logging
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
logging.getLogger("hmmlearn").setLevel(logging.ERROR)

# Resolve project root whether Jupyter starts in the repo root or notebooks/.
CWD = Path.cwd().resolve()
for candidate in [CWD, *CWD.parents]:
    if (candidate / "config" / "settings.py").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError(
        f"Could not locate project root (no config/settings.py found in {CWD} or any parent)."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "factors"
RAW_DIR = PROJECT_ROOT / "data" / "raw"

# Required RAW artefacts (no derived files):
#  - prices.parquet  : adjusted-close stock panel (light option treats this as raw)
#  - vix.parquet     : VIX close (already raw)
#  - macro_fred.parquet : long-format raw FRED panel (no lag, no ffill, no z-score)
REQUIRED_RAW = {
    DATA_DIR / "prices.parquet": "Adjusted-close stock panel (raw under light option)",
    RAW_DIR / "macro_fred.parquet": "Raw FRED long panel (built by scripts/fetch_raw_macro.py)",
}
missing = [str(p) for p in REQUIRED_RAW if not p.exists()]
if missing:
    raise FileNotFoundError(
        f"Missing raw artefacts: {missing}. Run scripts/fetch_raw_macro.py "
        "and the price backfill before this notebook."
    )

print(f"Project root : {PROJECT_ROOT}")
print(f"Raw dir      : {RAW_DIR}")
print(f"Factors dir  : {DATA_DIR}")
for path, role in REQUIRED_RAW.items():
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"  - {path.name}: {size_mb:.1f} MB ({role})")


# ---------------------------------------------------------------------------
# Notebook-wide helpers reused across every variable below.
# ---------------------------------------------------------------------------

STRESS_WINDOWS = {
    "dot_com_aftershock": ("2001-01-01", "2002-12-31"),
    "global_financial_crisis": ("2008-09-01", "2009-03-31"),
    "covid_shock": ("2020-02-15", "2020-04-30"),
    "inflation_rate_shock": ("2022-01-01", "2022-10-31"),
}


def ensure_tz_naive_index(obj):
    out = obj.copy()
    if getattr(out.index, "tz", None) is not None:
        out.index = out.index.tz_localize(None)
    return out


def describe_series(series: pd.Series, name: str) -> pd.DataFrame:
    clean = series.dropna()
    return pd.DataFrame(
        {
            "value": {
                "name": name,
                "dtype": str(series.dtype),
                "rows": len(series),
                "non_null_rows": int(series.notna().sum()),
                "missing_share": float(series.isna().mean()) if len(series) else np.nan,
                "start": series.index.min() if len(series) else pd.NaT,
                "end": series.index.max() if len(series) else pd.NaT,
                "min": clean.min() if len(clean) else np.nan,
                "median": clean.median() if len(clean) else np.nan,
                "max": clean.max() if len(clean) else np.nan,
            }
        }
    )


def plot_series(series: pd.Series, title: str, ylabel: str = "value") -> None:
    fig, ax = plt.subplots(figsize=(14, 3.2))
    series.dropna().plot(ax=ax, linewidth=0.8, color="steelblue")
    for _label, (start, end) in STRESS_WINDOWS.items():
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.08)
    ax.set_title(title, loc="left")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


def display_variable_gate(name: str, checks: dict) -> pd.DataFrame:
    gate = pd.DataFrame.from_dict(checks, orient="index", columns=["passes"])
    gate.index.name = name
    display(gate)
    failed = gate[~gate["passes"]].index.tolist()
    assert not failed, f"{name} failed gate(s): {failed}"
    return gate


## 2. Build features one variable at a time

The feature matrix the HMM will eventually see has nine columns. This section
builds them one at a time, in the strict pattern:

```text
load raw  ->  chart raw  ->  transform  ->  chart transformed  ->  ... ->  gate
```

Nothing in this section is hidden behind a single "build_regime_features" call.
Each transformation step lives in its own cell with its own chart so the reader
can audit it directly.

The raw inputs come from two places:

- `data/factors/prices.parquet` (adjusted-close stock panel; treated as raw under the light option for stock data).
- `data/raw/macro_fred.parquet` (long-format FRED panel, no publication lag, no business-day fill, no z-score).
- `core.data.vix.load_vix()` for VIX (already raw).

Variables produced and their source layer:

| Variable | Source | Built in |
|---|---|---|
| `mkt_returns` | prices | 2.0 |
| `factors` panel (vol_60d, beta_60d, mom_*) | prices | 2.0 |
| `return_dispersion` | prices | 2.1 |
| `beta_60d_spread` | factors | 2.2 |
| `vol_60d_median` | factors | 2.3 |
| `vix` | core.data.vix.load_vix | 2.4 |
| `macro_z_cpi_yoy` | raw FRED CPIAUCSL pct_change(12) | 2.5 |
| `macro_z_unrate` | raw FRED UNRATE | 2.6 |
| `macro_z_fed_funds` | raw FRED FEDFUNDS | 2.7 |
| `macro_z_dgs10` | raw FRED DGS10 | 2.8 |
| `macro_z_t10y2y` | raw FRED T10Y2Y | 2.9 |
| `features` (assembled) | concat all of the above | 2.10 |


### 2.0 Load raw artefacts

This cell loads the three raw artefacts and exposes a `get_raw_macro_series`
helper used by Sections 2.5-2.9. No transformation happens here yet — every
later sub-section pulls one variable from these raw containers and transforms
it inline.

In [ ]:
from core.data.factors.macro import (
    DEFAULT_FRED_SERIES_MAP,
    MACRO_PUBLICATION_LAGS_DAYS,
    RAW_MACRO_PARQUET,
    load_raw_macro_default,
)
from core.data.vix import load_vix

# Raw stock-price panel (treated as raw under light option).
upstream_prices = pd.read_parquet(DATA_DIR / "prices.parquet")
upstream_prices = ensure_tz_naive_index(upstream_prices)

# Raw VIX close from yfinance cache.
upstream_vix = ensure_tz_naive_index(load_vix()).sort_index().rename("upstream_vix")

# Raw FRED long panel; fetch from FRED if missing.
RAW_DIR.mkdir(parents=True, exist_ok=True)
if not RAW_MACRO_PARQUET.exists():
    print(f"Raw macro parquet not found. Fetching FRED -> {RAW_MACRO_PARQUET}")
    fresh_raw = load_raw_macro_default()
    if fresh_raw.empty:
        raise RuntimeError("FRED returned no rows; check FRED_API_KEY before re-running.")
    fresh_raw.to_parquet(RAW_MACRO_PARQUET, index=False)
raw_macro_long = pd.read_parquet(RAW_MACRO_PARQUET)
raw_macro_long["reference_date"] = pd.to_datetime(raw_macro_long["reference_date"])

raw_macro_summary = (
    raw_macro_long.groupby("series_id")["reference_date"]
    .agg(["min", "max", "count"])
    .reset_index()
    .rename(columns={"min": "first_obs", "max": "last_obs", "count": "n_observations"})
)
print(f"upstream_prices: {upstream_prices.shape}, "
      f"{upstream_prices.index.min().date()} .. {upstream_prices.index.max().date()}")
print(f"upstream_vix   : {len(upstream_vix)} rows, "
      f"{upstream_vix.index.min().date()} .. {upstream_vix.index.max().date()}")
print(f"raw_macro_long : {len(raw_macro_long):,} rows, {raw_macro_long['series_id'].nunique()} series")
display(raw_macro_summary)


def get_raw_macro_series(series_id: str) -> pd.Series:
    """Return one raw FRED series at native frequency (no lag, no ffill)."""
    rows = raw_macro_long.loc[raw_macro_long["series_id"] == series_id]
    if rows.empty:
        raise KeyError(f"series_id {series_id!r} not present in raw_macro_long")
    series = (
        rows[["reference_date", "value"]]
        .set_index("reference_date")
        .sort_index()["value"]
        .astype("float64")
    )
    series.index.name = "date"
    series.name = series_id
    return series


# Hygiene: blank single-day artefact returns above 300%, then market returns.
_MAX_DAILY_RETURN = 3.0
upstream_returns = upstream_prices.pct_change(fill_method=None)
bad_cells = upstream_returns.abs() > _MAX_DAILY_RETURN
n_bad = int(bad_cells.to_numpy().sum())
prices = upstream_prices.mask(bad_cells)
mkt_returns = prices.pct_change(fill_method=None).mean(axis=1).dropna()
mkt_returns.name = "market"
print(f"Price hygiene: blanked {n_bad} cells with |daily return| > {_MAX_DAILY_RETURN*100:.0f}%")
print(f"prices (clean) : {prices.shape}")
print(f"mkt_returns    : {len(mkt_returns)} rows, "
      f"{mkt_returns.index.min().date()} .. {mkt_returns.index.max().date()}")

# Per-stock factor panel rebuilt inline from raw prices (no factors_price.parquet).
stock_returns_wide = prices.pct_change(fill_method=None)
vol_60d_wide = stock_returns_wide.rolling(60, min_periods=40).std() * np.sqrt(252)
mean_stock_60 = stock_returns_wide.rolling(60, min_periods=40).mean()
mean_market_60 = mkt_returns.rolling(60, min_periods=40).mean()
mean_cross_60 = stock_returns_wide.mul(mkt_returns, axis=0).rolling(60, min_periods=40).mean()
cov_60d_wide = mean_cross_60.sub(mean_stock_60.mul(mean_market_60, axis=0))
var_market_60 = mkt_returns.rolling(60, min_periods=40).var()
beta_60d_wide = cov_60d_wide.div(var_market_60, axis=0)
mom_12_1_wide = prices.shift(21) / prices.shift(252) - 1
mom_6_1_wide = prices.shift(21) / prices.shift(126) - 1
mom_3_1_wide = prices.shift(21) / prices.shift(63) - 1


def _stack(wide_df, name):
    stacked = wide_df.stack(dropna=True).rename(name)
    stacked.index = stacked.index.set_names(["date", "symbol"])
    return stacked


factors = pd.concat(
    [
        _stack(vol_60d_wide, "vol_60d"),
        _stack(beta_60d_wide, "beta_60d"),
        _stack(mom_12_1_wide, "mom_12_1"),
        _stack(mom_6_1_wide, "mom_6_1"),
        _stack(mom_3_1_wide, "mom_3_1"),
    ],
    axis=1,
).sort_index()

date_level = factors.index.get_level_values("date")
if getattr(date_level, "tz", None) is not None:
    factors.index = factors.index.set_levels(date_level.tz_localize(None), level="date")

print(f"factors panel : {factors.shape}, columns = {factors.columns.tolist()}")

ZSCORE_WINDOW = 252 * 5
ZSCORE_MIN_PERIODS = max(30, ZSCORE_WINDOW // 12)
ALIGN_FFILL_LIMIT_DAYS = 21


### 2.1 return_dispersion

- **Source**: cleaned `prices` panel (Section 2.0).
- **Transformation**: per-stock daily arithmetic returns, then cross-sectional standard deviation across symbols on each business day.
- **Missing-data handling**: stocks without a return on a given day are excluded from that day's std; all-NaN days drop out at the feature-matrix step.
- **Publication lag**: none. Adjusted closes are observed end-of-day.
- **Standardisation**: walk-forward StandardScaler inside `fit_regime_hmm`; this notebook keeps it in raw cross-sectional std units.
- **Leakage risk**: low. Same-day cross-sectional summary, no future closes.

In [ ]:
raw_return_dispersion = (
    prices.pct_change(fill_method=None).std(axis=1).rename("raw_return_dispersion")
)
display(describe_series(raw_return_dispersion, "raw_return_dispersion"))
plot_series(raw_return_dispersion, "raw_return_dispersion (cross-sectional std of daily returns)", "std")

In [ ]:
return_dispersion = raw_return_dispersion.asfreq("B").rename("return_dispersion")
plot_series(return_dispersion, "return_dispersion (business-day aligned)", "std")
display_variable_gate(
    "return_dispersion",
    {
        "has_rows": return_dispersion.notna().sum() > 252,
        "non_negative": bool(return_dispersion.dropna().ge(0).all()),
        "finite_values": bool(np.isfinite(return_dispersion.dropna()).all()),
        "stress_elevated": float(
            return_dispersion.loc["2008-09-01":"2009-03-31"].mean()
        ) > float(return_dispersion.quantile(0.75)),
    },
)

### 2.2 beta_60d_spread

- **Source**: `factors["beta_60d"]` (Section 2.0), itself rebuilt inline as 60-day rolling `Cov(r_i, r_m) / Var(r_m)` per stock.
- **Transformation**: cross-sectional top-decile minus bottom-decile of `beta_60d` on each date.
- **Missing-data handling**: NaN per-stock betas are dropped before the daily quantiles; days with too few names produce NaN at this step.
- **Publication lag**: none beyond the trailing 60-day window already in `beta_60d`.
- **Standardisation**: walk-forward inside `fit_regime_hmm`; raw value here is dimensionless.
- **Leakage risk**: low. Trailing inputs only.

In [ ]:
raw_beta_60d_median = (
    factors["beta_60d"].dropna().groupby(level="date").median().rename("raw_beta_60d_median")
)
display(describe_series(raw_beta_60d_median, "raw_beta_60d_median"))
plot_series(raw_beta_60d_median, "raw_beta_60d (cross-sectional median)", "beta")

In [ ]:
_beta_groups = factors["beta_60d"].dropna().groupby(level="date")
raw_beta_60d_spread = (
    _beta_groups.quantile(0.90) - _beta_groups.quantile(0.10)
).rename("raw_beta_60d_spread")
plot_series(raw_beta_60d_spread, "raw_beta_60d_spread (top decile - bottom decile)", "beta")

In [ ]:
beta_60d_spread = raw_beta_60d_spread.asfreq("B").rename("beta_60d_spread")
plot_series(beta_60d_spread, "beta_60d_spread (business-day aligned)", "beta")
display_variable_gate(
    "beta_60d_spread",
    {
        "has_rows": beta_60d_spread.notna().sum() > 252,
        "non_negative": bool(beta_60d_spread.dropna().ge(0).all()),
        "finite_values": bool(np.isfinite(beta_60d_spread.dropna()).all()),
        "not_constant": float(beta_60d_spread.dropna().std()) > 0,
    },
)

### 2.3 vol_60d_median

- **Source**: `factors["vol_60d"]` (Section 2.0), itself a per-stock 60-day rolling std of daily returns annualised by `sqrt(252)`.
- **Transformation**: cross-sectional median across symbols on each date.
- **Missing-data handling**: NaN per-stock vol drops before the daily median.
- **Publication lag**: none beyond the trailing 60-day window already in `vol_60d`.
- **Standardisation**: walk-forward inside `fit_regime_hmm`.
- **Leakage risk**: low.

In [ ]:
raw_vol_60d_median = (
    factors["vol_60d"].dropna().groupby(level="date").median().rename("raw_vol_60d_median")
)
display(describe_series(raw_vol_60d_median, "raw_vol_60d_median"))
plot_series(raw_vol_60d_median, "raw_vol_60d (cross-sectional median)", "annualised std")

In [ ]:
vol_60d_median = raw_vol_60d_median.asfreq("B").rename("vol_60d_median")
plot_series(vol_60d_median, "vol_60d_median (business-day aligned)", "annualised std")
display_variable_gate(
    "vol_60d_median",
    {
        "has_rows": vol_60d_median.notna().sum() > 252,
        "non_negative": bool(vol_60d_median.dropna().ge(0).all()),
        "finite_values": bool(np.isfinite(vol_60d_median.dropna()).all()),
        "stress_elevated": float(
            vol_60d_median.loc["2008-09-01":"2009-03-31"].mean()
        ) > float(vol_60d_median.quantile(0.75)),
    },
)

### 2.4 vix

- **Source**: `core.data.vix.load_vix()` (yfinance cache, `data/factors/vix.parquet`). Already raw.
- **Transformation**: business-day reindex with 5-day forward-fill to bridge holiday gaps.
- **Missing-data handling**: 5-day ffill cap so a long market closure does not silently propagate.
- **Publication lag**: none. End-of-day observable.
- **Standardisation**: walk-forward inside `fit_regime_hmm`; raw level here is in index points.
- **Leakage risk**: low.

In [ ]:
display(describe_series(upstream_vix, "upstream_vix"))
plot_series(upstream_vix, "upstream_vix (raw VIX close)", "index level")

In [ ]:
vix = upstream_vix.asfreq("B").ffill(limit=5).rename("vix")
plot_series(vix, "vix (business-day aligned, 5d ffill)", "index level")
display_variable_gate(
    "vix",
    {
        "has_rows": vix.notna().sum() > 252,
        "non_negative": bool(vix.dropna().ge(0).all()),
        "finite_values": bool(np.isfinite(vix.dropna()).all()),
        "stress_spikes_visible": float(vix.loc["2008-09-01":"2009-03-31"].max())
        > float(vix.quantile(0.90)),
    },
)

### 2.5 macro_z_cpi_yoy

- **Source**: `data/raw/macro_fred.parquet`, `series_id == "cpi_yoy"`. Built from FRED `CPIAUCSL.pct_change(12)`.
- **Native frequency**: monthly.
- **Units**: YoY fraction (0.025 = 2.5%).
- **Publication lag applied here**: `+30d` calendar days. CPI prints ~2-3 weeks after the reference month; 30d is conservative.
- **Business-day alignment**: `asfreq("B").ffill(limit=21)` so a stale value persists for at most one month between releases.
- **Standardisation**: 5-year (1260 BD) rolling z-score with `min_periods=105`. Trailing-only, no future leakage.
- **Leakage risk**: low. Reference-date input + conservative pub-lag + trailing rolling stats; the HMM applies its own walk-forward standardisation downstream.

In [ ]:
raw_cpi_yoy = get_raw_macro_series("cpi_yoy")
display(describe_series(raw_cpi_yoy, "raw_cpi_yoy"))
plot_series(raw_cpi_yoy, "raw_cpi_yoy (FRED reference dates, no lag, no ffill)", "YoY fraction (0.025 = 2.5%)")

In [ ]:
lagged_cpi_yoy = raw_cpi_yoy.copy()
lagged_cpi_yoy.index = lagged_cpi_yoy.index + pd.to_timedelta(
    MACRO_PUBLICATION_LAGS_DAYS["cpi_yoy"], unit="D"
)
lagged_cpi_yoy = lagged_cpi_yoy.rename("lagged_cpi_yoy")
plot_series(lagged_cpi_yoy, "lagged_cpi_yoy (raw + 30d publication lag)", "YoY fraction (0.025 = 2.5%)")

In [ ]:
aligned_cpi_yoy = (
    lagged_cpi_yoy.asfreq("B").ffill(limit=ALIGN_FFILL_LIMIT_DAYS).rename("aligned_cpi_yoy")
)
plot_series(aligned_cpi_yoy, "aligned_cpi_yoy (business-day, 21d ffill)", "YoY fraction (0.025 = 2.5%)")

In [ ]:
roll_cpi_yoy = aligned_cpi_yoy.rolling(ZSCORE_WINDOW, min_periods=ZSCORE_MIN_PERIODS)
macro_z_cpi_yoy = (
    (aligned_cpi_yoy - roll_cpi_yoy.mean())
    / roll_cpi_yoy.std(ddof=0).replace(0, np.nan)
).rename("macro_z_cpi_yoy")
plot_series(macro_z_cpi_yoy, "macro_z_cpi_yoy (5y rolling z-score)", "z-score")
display_variable_gate(
    "macro_z_cpi_yoy",
    {
        "has_rows": macro_z_cpi_yoy.notna().sum() > 252,
        "finite_values": bool(np.isfinite(macro_z_cpi_yoy.dropna()).all()),
        "not_constant": float(macro_z_cpi_yoy.dropna().std()) > 0,
    },
)

### 2.6 macro_z_unrate

- **Source**: `data/raw/macro_fred.parquet`, `series_id == "unrate"`. Built from FRED `UNRATE`.
- **Native frequency**: monthly.
- **Units**: percent.
- **Publication lag applied here**: `+10d` calendar days. BLS releases ~1 week into the next month; 10d is conservative.
- **Business-day alignment**: `asfreq("B").ffill(limit=21)` so a stale value persists for at most one month between releases.
- **Standardisation**: 5-year (1260 BD) rolling z-score with `min_periods=105`. Trailing-only, no future leakage.
- **Leakage risk**: low. Reference-date input + conservative pub-lag + trailing rolling stats; the HMM applies its own walk-forward standardisation downstream.

In [ ]:
raw_unrate = get_raw_macro_series("unrate")
display(describe_series(raw_unrate, "raw_unrate"))
plot_series(raw_unrate, "raw_unrate (FRED reference dates, no lag, no ffill)", "percent")

In [ ]:
lagged_unrate = raw_unrate.copy()
lagged_unrate.index = lagged_unrate.index + pd.to_timedelta(
    MACRO_PUBLICATION_LAGS_DAYS["unrate"], unit="D"
)
lagged_unrate = lagged_unrate.rename("lagged_unrate")
plot_series(lagged_unrate, "lagged_unrate (raw + 10d publication lag)", "percent")

In [ ]:
aligned_unrate = (
    lagged_unrate.asfreq("B").ffill(limit=ALIGN_FFILL_LIMIT_DAYS).rename("aligned_unrate")
)
plot_series(aligned_unrate, "aligned_unrate (business-day, 21d ffill)", "percent")

In [ ]:
roll_unrate = aligned_unrate.rolling(ZSCORE_WINDOW, min_periods=ZSCORE_MIN_PERIODS)
macro_z_unrate = (
    (aligned_unrate - roll_unrate.mean())
    / roll_unrate.std(ddof=0).replace(0, np.nan)
).rename("macro_z_unrate")
plot_series(macro_z_unrate, "macro_z_unrate (5y rolling z-score)", "z-score")
display_variable_gate(
    "macro_z_unrate",
    {
        "has_rows": macro_z_unrate.notna().sum() > 252,
        "finite_values": bool(np.isfinite(macro_z_unrate.dropna()).all()),
        "not_constant": float(macro_z_unrate.dropna().std()) > 0,
    },
)

### 2.7 macro_z_fed_funds

- **Source**: `data/raw/macro_fred.parquet`, `series_id == "fed_funds"`. Built from FRED `FEDFUNDS`.
- **Native frequency**: monthly.
- **Units**: percent.
- **Publication lag applied here**: `+5d` calendar days. Monthly average is published shortly after month-end; 5d covers release timing.
- **Business-day alignment**: `asfreq("B").ffill(limit=21)` so a stale value persists for at most one month between releases.
- **Standardisation**: 5-year (1260 BD) rolling z-score with `min_periods=105`. Trailing-only, no future leakage.
- **Leakage risk**: low. Reference-date input + conservative pub-lag + trailing rolling stats; the HMM applies its own walk-forward standardisation downstream.

In [ ]:
raw_fed_funds = get_raw_macro_series("fed_funds")
display(describe_series(raw_fed_funds, "raw_fed_funds"))
plot_series(raw_fed_funds, "raw_fed_funds (FRED reference dates, no lag, no ffill)", "percent")

In [ ]:
lagged_fed_funds = raw_fed_funds.copy()
lagged_fed_funds.index = lagged_fed_funds.index + pd.to_timedelta(
    MACRO_PUBLICATION_LAGS_DAYS["fed_funds"], unit="D"
)
lagged_fed_funds = lagged_fed_funds.rename("lagged_fed_funds")
plot_series(lagged_fed_funds, "lagged_fed_funds (raw + 5d publication lag)", "percent")

In [ ]:
aligned_fed_funds = (
    lagged_fed_funds.asfreq("B").ffill(limit=ALIGN_FFILL_LIMIT_DAYS).rename("aligned_fed_funds")
)
plot_series(aligned_fed_funds, "aligned_fed_funds (business-day, 21d ffill)", "percent")

In [ ]:
roll_fed_funds = aligned_fed_funds.rolling(ZSCORE_WINDOW, min_periods=ZSCORE_MIN_PERIODS)
macro_z_fed_funds = (
    (aligned_fed_funds - roll_fed_funds.mean())
    / roll_fed_funds.std(ddof=0).replace(0, np.nan)
).rename("macro_z_fed_funds")
plot_series(macro_z_fed_funds, "macro_z_fed_funds (5y rolling z-score)", "z-score")
display_variable_gate(
    "macro_z_fed_funds",
    {
        "has_rows": macro_z_fed_funds.notna().sum() > 252,
        "finite_values": bool(np.isfinite(macro_z_fed_funds.dropna()).all()),
        "not_constant": float(macro_z_fed_funds.dropna().std()) > 0,
    },
)

### 2.8 macro_z_dgs10

- **Source**: `data/raw/macro_fred.parquet`, `series_id == "dgs10"`. Built from FRED `DGS10`.
- **Native frequency**: daily.
- **Units**: percent.
- **Publication lag applied here**: `+1d` calendar days. End-of-day observable; 1d lag avoids same-day leakage.
- **Business-day alignment**: `asfreq("B").ffill(limit=21)` so a stale value persists for at most one month between releases.
- **Standardisation**: 5-year (1260 BD) rolling z-score with `min_periods=105`. Trailing-only, no future leakage.
- **Leakage risk**: low. Reference-date input + conservative pub-lag + trailing rolling stats; the HMM applies its own walk-forward standardisation downstream.

In [ ]:
raw_dgs10 = get_raw_macro_series("dgs10")
display(describe_series(raw_dgs10, "raw_dgs10"))
plot_series(raw_dgs10, "raw_dgs10 (FRED reference dates, no lag, no ffill)", "percent")

In [ ]:
lagged_dgs10 = raw_dgs10.copy()
lagged_dgs10.index = lagged_dgs10.index + pd.to_timedelta(
    MACRO_PUBLICATION_LAGS_DAYS["dgs10"], unit="D"
)
lagged_dgs10 = lagged_dgs10.rename("lagged_dgs10")
plot_series(lagged_dgs10, "lagged_dgs10 (raw + 1d publication lag)", "percent")

In [ ]:
aligned_dgs10 = (
    lagged_dgs10.asfreq("B").ffill(limit=ALIGN_FFILL_LIMIT_DAYS).rename("aligned_dgs10")
)
plot_series(aligned_dgs10, "aligned_dgs10 (business-day, 21d ffill)", "percent")

In [ ]:
roll_dgs10 = aligned_dgs10.rolling(ZSCORE_WINDOW, min_periods=ZSCORE_MIN_PERIODS)
macro_z_dgs10 = (
    (aligned_dgs10 - roll_dgs10.mean())
    / roll_dgs10.std(ddof=0).replace(0, np.nan)
).rename("macro_z_dgs10")
plot_series(macro_z_dgs10, "macro_z_dgs10 (5y rolling z-score)", "z-score")
display_variable_gate(
    "macro_z_dgs10",
    {
        "has_rows": macro_z_dgs10.notna().sum() > 252,
        "finite_values": bool(np.isfinite(macro_z_dgs10.dropna()).all()),
        "not_constant": float(macro_z_dgs10.dropna().std()) > 0,
    },
)

### 2.9 macro_z_t10y2y

- **Source**: `data/raw/macro_fred.parquet`, `series_id == "t10y2y"`. Built from FRED `T10Y2Y`.
- **Native frequency**: daily.
- **Units**: percentage points.
- **Publication lag applied here**: `+1d` calendar days. End-of-day observable; 1d lag avoids same-day leakage.
- **Business-day alignment**: `asfreq("B").ffill(limit=21)` so a stale value persists for at most one month between releases.
- **Standardisation**: 5-year (1260 BD) rolling z-score with `min_periods=105`. Trailing-only, no future leakage.
- **Leakage risk**: low. Reference-date input + conservative pub-lag + trailing rolling stats; the HMM applies its own walk-forward standardisation downstream.

In [ ]:
raw_t10y2y = get_raw_macro_series("t10y2y")
display(describe_series(raw_t10y2y, "raw_t10y2y"))
plot_series(raw_t10y2y, "raw_t10y2y (FRED reference dates, no lag, no ffill)", "percentage points")

In [ ]:
lagged_t10y2y = raw_t10y2y.copy()
lagged_t10y2y.index = lagged_t10y2y.index + pd.to_timedelta(
    MACRO_PUBLICATION_LAGS_DAYS["t10y2y"], unit="D"
)
lagged_t10y2y = lagged_t10y2y.rename("lagged_t10y2y")
plot_series(lagged_t10y2y, "lagged_t10y2y (raw + 1d publication lag)", "percentage points")

In [ ]:
aligned_t10y2y = (
    lagged_t10y2y.asfreq("B").ffill(limit=ALIGN_FFILL_LIMIT_DAYS).rename("aligned_t10y2y")
)
plot_series(aligned_t10y2y, "aligned_t10y2y (business-day, 21d ffill)", "percentage points")

In [ ]:
roll_t10y2y = aligned_t10y2y.rolling(ZSCORE_WINDOW, min_periods=ZSCORE_MIN_PERIODS)
macro_z_t10y2y = (
    (aligned_t10y2y - roll_t10y2y.mean())
    / roll_t10y2y.std(ddof=0).replace(0, np.nan)
).rename("macro_z_t10y2y")
plot_series(macro_z_t10y2y, "macro_z_t10y2y (5y rolling z-score)", "z-score")
display_variable_gate(
    "macro_z_t10y2y",
    {
        "has_rows": macro_z_t10y2y.notna().sum() > 252,
        "finite_values": bool(np.isfinite(macro_z_t10y2y.dropna()).all()),
        "not_constant": float(macro_z_t10y2y.dropna().std()) > 0,
    },
)

### 2.10 Assemble feature matrix

Concat the nine per-variable Series into one wide DataFrame on a business-day
index, drop the leading rows where any input is still NaN, and gate that the
matrix has the expected schema and no missing rows.

In [ ]:
features = pd.concat(
    [
        return_dispersion,
        beta_60d_spread,
        vol_60d_median,
        macro_z_t10y2y,
        macro_z_fed_funds,
        macro_z_unrate,
        macro_z_dgs10,
        macro_z_cpi_yoy,
        vix,
    ],
    axis=1,
).asfreq("B").ffill().dropna()

print(f"Feature matrix: {features.shape}")
print(f"Columns: {features.columns.tolist()}")
display(features.describe().round(4))

display_variable_gate(
    "features",
    {
        "has_rows": len(features) > 252 * 5,
        "has_expected_columns": features.columns.tolist() == [
            "return_dispersion",
            "beta_60d_spread",
            "vol_60d_median",
            "macro_z_t10y2y",
            "macro_z_fed_funds",
            "macro_z_unrate",
            "macro_z_dgs10",
            "macro_z_cpi_yoy",
            "vix",
        ],
        "no_missing_rows": not bool(features.isna().any().any()),
        "monotonic_unique_index": bool(
            features.index.is_monotonic_increasing and features.index.is_unique
        ),
    },
)

## 3. Walk-Forward HMM Estimation

Now we ask the HMM to infer hidden regimes from the feature matrix.

The model does not start with labels like "calm" or "crisis." It starts with anonymous states. We label those states only after fitting, using the trailing equal-weighted market-return behavior observed inside the training window:

- highest mean `mkt_returns` state becomes `risk_on`;
- middle state becomes `neutral`;
- lowest mean `mkt_returns` state becomes `risk_off`.

That means a sentence like "today looks high-risk" has a precise meaning here: today's feature pattern, after being standardized with trailing-only information, received a high filtered probability for the state that historically had the weakest market-return behavior in the current training window.

Important implementation details:

- The model is fit walk-forward. Each prediction block uses the trailing 5 years of features for both scaling and HMM fitting.
- The fitted scaler, HMM parameters, and state-label mapping are held fixed inside each 21-day prediction block.
- Probabilities still update daily because each day's observed features are added to the filtered probability calculation.
- The output probabilities are model beliefs, not ground truth labels. A high `p_risk_off` means "observations through today resemble historical risk-off states."
- The notebook uses filtered probabilities, not smoothed probabilities, so a date's probability does not use later observations from the same prediction block.

What would support the idea:

- regimes persist for stretches rather than flipping randomly every day;
- `risk_off` clusters around obvious stress periods;
- `risk_off` has weaker market returns or higher realized volatility than `risk_on` in the diagnostics;
- probability changes occur near meaningful market transitions.

What would weaken the idea:

- labels are evenly random or unstable;
- probabilities are always extreme without matching market history;
- `risk_off` does not look economically worse than `risk_on`;
- detected states have no relationship to returns, downside risk, or drawdowns.

In [ ]:
from core.signals.regime_hmm import fit_regime_hmm

regime_probs = fit_regime_hmm(
    features,
    n_states=3,
    train_window=252 * 5,
    step=21,
    market_returns=mkt_returns,
    predict_mode="filtered",
)

prob_cols = [col for col in regime_probs.columns if col.startswith("p_")]
regime_distribution = regime_probs["regime"].value_counts().rename("days")
regime_share = regime_probs["regime"].value_counts(normalize=True).rename("share")
regime_summary = pd.concat([regime_distribution, regime_share], axis=1)

print(f"Regime probs: {regime_probs.shape}")
display(regime_summary)
regime_probs.head(10)

In [ ]:
market_equity = (1 + mkt_returns).cumprod()
market_equity = market_equity.reindex(regime_probs.index).ffill()
market_equity = market_equity / market_equity.dropna().iloc[0]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                                gridspec_kw={"height_ratios": [2, 1]})

ax1.plot(market_equity.index, market_equity.values, color="black",
         linewidth=0.8, label="Market equity (equal-weighted)")
ymin, ymax = market_equity.min(), market_equity.max()
for regime, color in [("risk_off", "#e74c3c"), ("neutral", "#f39c12"), ("risk_on", "#27ae60")]:
    mask = regime_probs["regime"] == regime
    if mask.any():
        ax1.fill_between(regime_probs.index, ymin, ymax,
                         where=mask.values, alpha=0.15, color=color, label=regime)
ax1.set_ylabel("Equity (normalized)")
ax1.set_title("Equal-Weighted Market Equity with Detected Regimes")
ax1.legend(loc="upper left")
ax1.grid(alpha=0.3)

prob_cols = [c for c in regime_probs.columns if c.startswith("p_")]
color_map = {"p_risk_on": "#27ae60", "p_neutral": "#f39c12", "p_risk_off": "#e74c3c"}
regime_probs[prob_cols].plot.area(ax=ax2, stacked=True,
    color=[color_map.get(c, "gray") for c in prob_cols], alpha=0.7, linewidth=0)
ax2.set_ylabel("Probability")
ax2.set_title("State Probabilities")
ax2.set_ylim(0, 1)
ax2.legend(loc="upper left")
ax2.grid(alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
regime_return_context = pd.DataFrame(index=regime_probs.index)
regime_return_context["regime"] = regime_probs["regime"]
regime_return_context["market_return_bps"] = mkt_returns.reindex(regime_probs.index) * 1e4
regime_return_context["market_vol_21d_ann"] = (
    mkt_returns.reindex(regime_probs.index).rolling(21).std() * np.sqrt(252)
)

regime_diagnostics = (
    regime_return_context.groupby("regime")
    .agg(
        days=("market_return_bps", "size"),
        mean_market_return_bps=("market_return_bps", "mean"),
        median_market_return_bps=("market_return_bps", "median"),
        mean_market_vol_21d_ann=("market_vol_21d_ann", "mean"),
    )
    .sort_values("mean_market_return_bps", ascending=False)
)
regime_diagnostics.round(4)

In [ ]:
regime_confidence = regime_probs[prob_cols].max(axis=1).rename("max_regime_probability")
confidence_by_regime = (
    pd.DataFrame(
        {
            "regime": regime_probs["regime"],
            "max_regime_probability": regime_confidence,
        }
    )
    .groupby("regime")
    .agg(
        days=("max_regime_probability", "size"),
        avg_max_probability=("max_regime_probability", "mean"),
        median_max_probability=("max_regime_probability", "median"),
        share_above_95pct=("max_regime_probability", lambda values: (values > 0.95).mean()),
        share_above_99pct=("max_regime_probability", lambda values: (values > 0.99).mean()),
    )
    .sort_values("avg_max_probability", ascending=False)
)
confidence_by_regime.round(3)

In [ ]:
confidence_by_year = (
    regime_confidence.groupby(regime_confidence.index.year)
    .agg(
        avg_max_probability="mean",
        share_above_95pct=lambda values: (values > 0.95).mean(),
        share_above_99pct=lambda values: (values > 0.99).mean(),
    )
    .rename_axis("year")
)
confidence_by_year.tail(10).round(3)

### HMM Interpretation

Use the regime distribution, market overlay, probability plot, diagnostics table, and confidence tables together. No single chart proves the regime labels are right.

The state names are credible only if the diagnostics support them:

- `risk_on` should generally have better average market returns or calmer realized volatility than `risk_off`.
- `risk_off` should show up around known selloffs or periods of elevated market stress.
- The probability chart should show persistent stretches, not random one-day flickering.

This is the transparency check behind the phrase "today looks high-risk." The model is not saying that because of a hidden story. It is saying that because today's standardized `return_dispersion`, `beta_60d_spread`, `vol_60d_median`, `macro_z_*`, and `vix` values look more like the training-window state that was labeled `risk_off`.

The confidence tables show how often the model assigns near-certain state probabilities. Very sharp probabilities, such as values extremely close to `0` or `1`, are not automatically good. They can mean the states are well separated, but they can also indicate overconfidence, unstable fits, or features with extreme outliers. If most days are above `0.99`, compare alternative state counts or covariance assumptions before trusting the signal.

## 4. Backtest: Regime-Scaled Vs Unscaled Momentum

The trading test is deliberately simple: keep the same `mom_12_1` long/short strategy, then multiply its daily return by yesterday's `p_risk_on`.

This is the exposure overlay:

```text
Momentum strategy says: this is today's long/short momentum return.
Regime model says: yesterday's probability of risk-on was X.
Overlay allows: X percent of the normal momentum exposure today.
```

So the allowed exposure is disclosed directly by the `exposure` series:

- `exposure = p_risk_on`, clipped to the range `0.0` to `1.0` by `regime_signal(..., mode="exposure_scale")`.
- `1.0` means hold 100% of the normal momentum exposure.
- `0.5` means hold 50% of the normal momentum exposure.
- `0.0` means hold none of the momentum exposure.
- The notebook shifts this exposure by one day before applying it, so the regime estimate from date `t` can only affect returns from date `t + 1` onward.

This tests whether the regime model is useful as a risk-management layer, not whether the HMM itself predicts stock returns. The overlay is helpful only if it improves the portfolio's downside-risk profile after accounting for lower exposure.

Expected behavior if the idea works:

- lower tail losses because exposure falls outside risk-on environments;
- smaller max drawdown because exposure is reduced during stressed regimes;
- better Sortino or Calmar, not just lower raw return and lower volatility.

A weaker result would be: the scaled portfolio looks safer only because it is smaller, while downside-adjusted metrics do not improve.

In [ ]:
from core.signals.regime_hmm import regime_signal
from core.strategies.factor_runner import run_factor_cross_section_backtest
from core.backtest.portfolio import sp500_universe_filter
from core.metrics.performance import (
    calculate_performance_metrics,
    calculate_drawdown,
)

exposure = regime_signal(regime_probs, mode="exposure_scale")
exposure_summary = exposure.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).rename(
    "allowed_exposure"
)
exposure_by_regime = (
    pd.DataFrame(
        {
            "regime": regime_probs["regime"],
            "allowed_exposure": exposure,
        }
    )
    .groupby("regime")
    .agg(
        days=("allowed_exposure", "size"),
        mean_allowed_exposure=("allowed_exposure", "mean"),
        median_allowed_exposure=("allowed_exposure", "median"),
        min_allowed_exposure=("allowed_exposure", "min"),
        max_allowed_exposure=("allowed_exposure", "max"),
    )
    .sort_values("mean_allowed_exposure", ascending=False)
)

print(f"Exposure signal: mean={exposure.mean():.3f}, std={exposure.std():.3f}")
print(f"Date range: {exposure.index[0].date()} .. {exposure.index[-1].date()}")
display(exposure_summary.round(3))
display(exposure_by_regime.round(3))

In [ ]:
# Scope the momentum backtest to the HMM signal window on a survivorship-free
# S&P 500 universe. Running mom_12_1 over the raw 1927-2026 / 835-symbol panel
# explodes the numbers once delistings are realised as -100% (Bug 4 fix) —
# there is simply no well-defined factor universe in the 1930s-1980s here.
#
# `run_factor_cross_section_backtest` applies the standard pipeline:
#   * `signal_lag_days=1` — factor at close(t-1) drives the weight earning r(t)
#   * `min_stocks=20` — dates with fewer valid ranks emit zero signals
#   * `sp500_universe_filter()` — signals are only emitted on names in the
#     historical S&P 500 constituent set on that date
uf = sp500_universe_filter()
backtest_start = pd.Timestamp(regime_probs.index[0])
backtest_end = pd.Timestamp(min(regime_probs.index[-1], prices.index[-1].tz_localize(None)))

unscaled = run_factor_cross_section_backtest(
    factors,
    prices,
    factor_col="mom_12_1",
    start=backtest_start,
    end=backtest_end,
    top_pct=0.2,
    bottom_pct=0.2,
    long_only=False,
    rebalance_freq="ME",
    transaction_cost=0.001,
    min_stocks=20,
    universe_filter=uf,
    signal_lag_days=1,
)
unscaled = unscaled.dropna()
if unscaled.index.tz is not None:
    unscaled.index = unscaled.index.tz_localize(None)
unscaled.name = "unscaled_mom"
print(
    f"Unscaled momentum: {len(unscaled)} return days, "
    f"{unscaled.index[0].date()} .. {unscaled.index[-1].date()}"
)
print(f"Daily mean return: {unscaled.mean():.6f}  (std {unscaled.std():.6f})")

In [ ]:
# Shift exposure by 1 day to avoid look-ahead: regime_probs on date t is
# inferred from features observed at close(t) and is only actionable from
# close(t+1) onwards. The momentum signal itself already uses `signal_lag_days=1`
# so the unscaled return on day t comes from a weight set on close(t-1); we
# multiply by exposure(t-1) for consistency.
exposure_lagged = exposure.shift(1)

common_idx = unscaled.index.intersection(exposure_lagged.dropna().index)
exposure_aligned = exposure_lagged.reindex(common_idx).ffill().fillna(0.5)
scaled = unscaled.loc[common_idx] * exposure_aligned
scaled.name = "regime_scaled_mom"
unscaled_common = unscaled.loc[common_idx]
unscaled_common.name = "unscaled_mom"

applied_exposure_summary = exposure_aligned.describe(
    percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]
).rename("applied_exposure")

print(f"Backtest period: {common_idx[0].date()} .. {common_idx[-1].date()} ({len(common_idx)} days)")
print(f"Mean exposure applied: {exposure_aligned.mean():.3f}")
display(applied_exposure_summary.round(3))

## 5. Performance Comparison

This is the decision table. It compares two return streams over the same dates:

- `Unscaled Momentum`: the long/short `mom_12_1` daily return without any regime overlay.
- `Regime-Scaled Momentum`: the same daily return multiplied by yesterday's allowed exposure.

The table leads with downside-risk metrics, not volatility, because the question is whether the overlay reduces real losses, not whether it makes the portfolio smaller.

Read each row as a yes/no question:

- `sortino_ratio`: did return improve relative to downside deviation?
- `max_drawdown`: did the overlay reduce the worst peak-to-trough loss?
- `calmar_ratio`: did return per unit of drawdown improve?
- `cvar_95` and `cvar_99`: did the average bad-day loss in the worst 5% and worst 1% of days become smaller?
- `time_underwater_days`: did the longest stretch below the prior equity peak shorten?
- `loss_probability_252d`: did the bootstrap-estimated probability of a 1-year loss fall?

Volatility and Sharpe stay in the table for context, but they do not decide the conclusion. Lower volatility alone is not proof that risk fell; it can simply mean the portfolio is smaller.

In [ ]:
LOSS_PROBABILITY_HORIZONS = (252,)
LOSS_PROBABILITY_BOOTSTRAPS = 2_000

metrics_unscaled = calculate_performance_metrics(
    unscaled_common,
    loss_probability_horizons=LOSS_PROBABILITY_HORIZONS,
    loss_probability_bootstraps=LOSS_PROBABILITY_BOOTSTRAPS,
)
metrics_scaled = calculate_performance_metrics(
    scaled,
    loss_probability_horizons=LOSS_PROBABILITY_HORIZONS,
    loss_probability_bootstraps=LOSS_PROBABILITY_BOOTSTRAPS,
)

comparison = pd.DataFrame({
    "Unscaled Momentum": metrics_unscaled,
    "Regime-Scaled Momentum": metrics_scaled,
}).T

comparison_display = comparison.copy()
percent_cols = [
    "total_return",
    "annualized_return",
    "annualized_volatility",
    "max_drawdown",
    "cvar_95",
    "cvar_99",
    "loss_probability_252d",
]
for c in percent_cols:
    if c in comparison_display.columns:
        comparison_display[c] = comparison_display[c].map(lambda x: f"{x:.2%}")
for c in ["sharpe_ratio", "sortino_ratio", "calmar_ratio"]:
    if c in comparison_display.columns:
        comparison_display[c] = comparison_display[c].map(lambda x: f"{x:.3f}")
if "time_underwater_days" in comparison_display.columns:
    comparison_display["time_underwater_days"] = comparison_display["time_underwater_days"].map(lambda x: f"{int(x)}")

display_cols = [
    "annualized_return",
    "sortino_ratio",
    "max_drawdown",
    "calmar_ratio",
    "cvar_95",
    "cvar_99",
    "time_underwater_days",
    "loss_probability_252d",
    "annualized_volatility",
    "sharpe_ratio",
]
comparison_display[[c for c in display_cols if c in comparison_display.columns]]

In [ ]:
from IPython.display import Markdown

raw_comparison = pd.DataFrame(
    {
        "unscaled": metrics_unscaled,
        "regime_scaled": metrics_scaled,
    }
)
raw_comparison["delta_scaled_minus_unscaled"] = (
    raw_comparison["regime_scaled"] - raw_comparison["unscaled"]
)

key_metric_order = [
    "annualized_return",
    "sortino_ratio",
    "max_drawdown",
    "calmar_ratio",
    "cvar_95",
    "cvar_99",
    "time_underwater_days",
    "loss_probability_252d",
    "annualized_volatility",
    "sharpe_ratio",
]
delta_table = raw_comparison.loc[
    [metric for metric in key_metric_order if metric in raw_comparison.index]
]
display(delta_table.round(4))

sortino_delta = raw_comparison.loc["sortino_ratio", "delta_scaled_minus_unscaled"]
drawdown_delta = raw_comparison.loc["max_drawdown", "delta_scaled_minus_unscaled"]
calmar_delta = raw_comparison.loc["calmar_ratio", "delta_scaled_minus_unscaled"]
cvar_95_delta = raw_comparison.loc["cvar_95", "delta_scaled_minus_unscaled"]
loss_probability_delta = raw_comparison.loc[
    "loss_probability_252d", "delta_scaled_minus_unscaled"
]

risk_controls_improved = (
    drawdown_delta > 0
    and cvar_95_delta < 0
    and loss_probability_delta <= 0
)
risk_adjusted_improved = sortino_delta > 0 and calmar_delta > 0
if risk_controls_improved and risk_adjusted_improved:
    conclusion = (
        "The current run supports the overlay on downside-risk terms: max drawdown, "
        "CVaR(95), one-year loss probability, Sortino, and Calmar improved versus "
        "unscaled momentum."
    )
elif risk_controls_improved:
    conclusion = (
        "The current run reduces downside risk, but risk-adjusted performance is mixed. "
        "Treat the overlay as provisional rather than proven."
    )
else:
    conclusion = (
        "The current run does not support the overlay on the downside-risk controls "
        "that motivate regime scaling."
    )

display(Markdown(f"**Generated conclusion:** {conclusion}"))

### Performance Interpretation

The delta table is `regime_scaled - unscaled` for each metric. The sign convention matters and is easy to misread, so it is spelled out:

- **Positive delta is good** for `annualized_return`, `sortino_ratio`, `calmar_ratio`, and `max_drawdown`. (`max_drawdown` is a negative number, so a positive delta means the scaled drawdown was less negative.)
- **Negative delta is good** for `cvar_95`, `cvar_99`, `time_underwater_days`, `loss_probability_252d`, and `annualized_volatility`.

The generated conclusion below the table only fires when the downside checks line up. It uses Sortino, Calmar, max drawdown, CVaR(95), and 1-year loss probability. Sharpe and volatility are reported but do not drive the verdict, because both can fall just because the strategy is now smaller.

## 6. Multi-Factor Overlay Comparison

Section 6 only answers "does the overlay help `mom_12_1`?" That is too narrow. A regime overlay that only helps one factor is more likely a coincidence than a real risk-control layer. So this section runs the same exposure rule on a small set of nearby factors and compares the results.

The overlay rule is identical for every factor: take the unscaled long/short return, multiply by yesterday's clipped `p_risk_on`, and compare to the same factor without the overlay.

The factors tested in this run:

- `mom_12_1`, `mom_6_1`, `mom_3_1`: long higher trailing momentum, short lower trailing momentum.
- `vol_60d`: long lower realized volatility, short higher realized volatility.
- `beta_60d`: long lower beta, short higher beta.

Backtest settings match section 5 so the comparison is apples-to-apples: S&P 500 point-in-time universe, monthly rebalance, long/short quintiles, 10 bps transaction cost, and `signal_lag_days=1`.

In [ ]:
factor_overlay_specs = [
    {"label": "mom_12_1", "factor_col": "mom_12_1", "rank_direction": 1},
    {"label": "mom_6_1", "factor_col": "mom_6_1", "rank_direction": 1},
    {"label": "mom_3_1", "factor_col": "mom_3_1", "rank_direction": 1},
    {"label": "low_vol_60d", "factor_col": "vol_60d", "rank_direction": -1},
    {"label": "low_beta_60d", "factor_col": "beta_60d", "rank_direction": -1},
]

available_factor_cols = set(factors.columns)
factor_overlay_rows = []

for spec in factor_overlay_specs:
    source_col = spec["factor_col"]
    if source_col not in available_factor_cols:
        continue

    backtest_factors = factors
    backtest_factor_col = source_col
    if spec["rank_direction"] < 0:
        backtest_factor_col = f"__{spec['label']}_rank"
        backtest_factors = factors.assign(
            **{backtest_factor_col: -factors[source_col]}
        )

    if source_col == "mom_12_1" and spec["rank_direction"] == 1:
        factor_unscaled = unscaled.copy()
    else:
        factor_unscaled = run_factor_cross_section_backtest(
            backtest_factors,
            prices,
            factor_col=backtest_factor_col,
            start=backtest_start,
            end=backtest_end,
            top_pct=0.2,
            bottom_pct=0.2,
            long_only=False,
            rebalance_freq="ME",
            transaction_cost=0.001,
            min_stocks=20,
            universe_filter=uf,
            signal_lag_days=1,
        ).dropna()
        if factor_unscaled.index.tz is not None:
            factor_unscaled.index = factor_unscaled.index.tz_localize(None)

    factor_idx = factor_unscaled.index.intersection(exposure_lagged.dropna().index)
    factor_exposure = exposure_lagged.reindex(factor_idx).ffill().fillna(0.5)
    factor_unscaled_common = factor_unscaled.loc[factor_idx]
    factor_scaled = factor_unscaled_common * factor_exposure

    unscaled_metrics = calculate_performance_metrics(
        factor_unscaled_common,
        loss_probability_horizons=LOSS_PROBABILITY_HORIZONS,
        loss_probability_bootstraps=LOSS_PROBABILITY_BOOTSTRAPS,
    )
    scaled_metrics = calculate_performance_metrics(
        factor_scaled,
        loss_probability_horizons=LOSS_PROBABILITY_HORIZONS,
        loss_probability_bootstraps=LOSS_PROBABILITY_BOOTSTRAPS,
    )

    factor_overlay_rows.append(
        {
            "factor": spec["label"],
            "source_column": source_col,
            "rank_direction": "high_is_long" if spec["rank_direction"] > 0 else "low_is_long",
            "days": len(factor_idx),
            "mean_exposure": factor_exposure.mean(),
            "return_delta": scaled_metrics["annualized_return"] - unscaled_metrics["annualized_return"],
            "sortino_delta": scaled_metrics["sortino_ratio"] - unscaled_metrics["sortino_ratio"],
            "max_drawdown_delta": scaled_metrics["max_drawdown"] - unscaled_metrics["max_drawdown"],
            "calmar_delta": scaled_metrics["calmar_ratio"] - unscaled_metrics["calmar_ratio"],
            "cvar_95_delta": scaled_metrics["cvar_95"] - unscaled_metrics["cvar_95"],
            "cvar_99_delta": scaled_metrics["cvar_99"] - unscaled_metrics["cvar_99"],
            "time_underwater_delta": scaled_metrics["time_underwater_days"] - unscaled_metrics["time_underwater_days"],
            "loss_probability_252d_delta": scaled_metrics["loss_probability_252d"] - unscaled_metrics["loss_probability_252d"],
            "volatility_delta": scaled_metrics["annualized_volatility"] - unscaled_metrics["annualized_volatility"],
            "sharpe_delta": scaled_metrics["sharpe_ratio"] - unscaled_metrics["sharpe_ratio"],
        }
    )

factor_overlay_results = pd.DataFrame(factor_overlay_rows).set_index("factor")
summary_cols = [
    "rank_direction",
    "days",
    "mean_exposure",
    "return_delta",
    "sortino_delta",
    "max_drawdown_delta",
    "calmar_delta",
    "cvar_95_delta",
    "cvar_99_delta",
    "time_underwater_delta",
    "loss_probability_252d_delta",
    "volatility_delta",
    "sharpe_delta",
]
factor_overlay_results[summary_cols].round(4)

In [ ]:
benefit_flags = factor_overlay_results.assign(
    improves_sortino=factor_overlay_results["sortino_delta"] > 0,
    improves_calmar=factor_overlay_results["calmar_delta"] > 0,
    reduces_cvar_95=factor_overlay_results["cvar_95_delta"] < 0,
    reduces_drawdown=factor_overlay_results["max_drawdown_delta"] > 0,
)
benefit_flags[
    [
        "improves_sortino",
        "improves_calmar",
        "reduces_cvar_95",
        "reduces_drawdown",
    ]
]

In [ ]:
benefit_score = benefit_flags[
    ["improves_sortino", "improves_calmar", "reduces_cvar_95", "reduces_drawdown"]
].sum(axis=1)
benefiting_factors = benefit_score[benefit_score >= 3].index.tolist()
partial_factors = benefit_score[(benefit_score > 0) & (benefit_score < 3)].index.tolist()
failed_factors = benefit_score[benefit_score == 0].index.tolist()

if benefiting_factors:
    benefit_text = ", ".join(benefiting_factors)
    conclusion = (
        f"The overlay is most compelling for: {benefit_text}. These factors improve "
        "on at least three of the four downside checks: Sortino, Calmar, CVaR(95), "
        "and max drawdown."
    )
elif partial_factors:
    partial_text = ", ".join(partial_factors)
    conclusion = (
        f"The overlay has mixed evidence for: {partial_text}. It helps some downside "
        "metrics, but does not clear enough checks to call the benefit broad-based."
    )
else:
    conclusion = (
        "The overlay does not show broad factor-level benefit in this run. Treat it as "
        "a momentum-specific research idea unless later robustness checks improve."
    )

if failed_factors:
    conclusion += f" Factors with no positive downside checks: {', '.join(failed_factors)}."

display(Markdown(f"**Multi-factor overlay interpretation:** {conclusion}"))

### Multi-Factor Interpretation

Read this as a breadth check, not a strategy ranking. The question is: does the same overlay help on more than one factor, or only on `mom_12_1`?

A factor counts as a real beneficiary only if the scaled version improves downside control and downside-adjusted return, not merely because the portfolio is smaller. The generated interpretation uses four checks per factor:

- `sortino_ratio` improved;
- `calmar_ratio` improved;
- `cvar_95` became less negative;
- `max_drawdown` became less negative.

Factors that pass at least three of those four checks are the strongest candidates for follow-up. If only one factor passes, the regime overlay is most likely factor-specific. If none pass, the overlay is not yet adding broad downside protection.

## 7. Baselines: Does The HMM Beat Simpler Regime Indicators?

A complicated model has to pay rent. If a transparent rule already does the same job, the HMM is not yet justified.

This section compares the HMM exposure overlay against two simple baselines that anyone can reproduce:

- `vix_threshold`: hold full exposure when VIX is low, no exposure when VIX is high, with a linear transition between the two thresholds. This is "stand down when option-implied stress is high."
- `ma200`: hold full exposure when the S&P 500 is above its trailing 200-day moving average, otherwise no exposure. This is "stand down when the trend is broken."

Both baselines use only causal information: today's exposure can use only data observable through yesterday's close. Like the HMM overlay, each baseline exposure is shifted by one day before it touches returns, so the comparison is apples-to-apples.

The HMM is considered to add value only if its overlay produces clearly better downside risk than these two simpler rules. "About the same" is not enough.

In [ ]:
from core.signals.regime_baselines import (
    moving_average_exposure,
    vix_threshold_exposure,
)


def apply_lagged_exposure(
    returns: pd.Series,
    exposure_series: pd.Series,
    default_exposure: float = 0.0,
) -> tuple[pd.Series, pd.Series]:
    """Apply yesterday's exposure to today's return series."""
    lagged = exposure_series.shift(1)
    common = returns.index.intersection(lagged.dropna().index)
    aligned_exposure = lagged.reindex(common).ffill().fillna(default_exposure)
    scaled_returns = returns.loc[common] * aligned_exposure
    return scaled_returns.rename("scaled_return"), aligned_exposure


vix_baseline = vix_threshold_exposure(vix, low=15.0, high=30.0)
ma200_baseline = moving_average_exposure(prices, market_symbol="^GSPC", window=200)

baseline_exposures = {
    "hmm_exposure": exposure,
    "vix_threshold": vix_baseline,
    "ma200": ma200_baseline,
}

baseline_return_candidates = {"unscaled": unscaled.copy()}
baseline_exposure_candidates = {
    "unscaled": pd.Series(1.0, index=unscaled.index, name="unscaled_exposure")
}
for label, exposure_series in baseline_exposures.items():
    overlay_returns, aligned_exposure = apply_lagged_exposure(
        unscaled,
        exposure_series,
        default_exposure=0.0,
    )
    baseline_return_candidates[label] = overlay_returns
    baseline_exposure_candidates[label] = aligned_exposure

baseline_common_idx = baseline_return_candidates["unscaled"].index
for returns_series in baseline_return_candidates.values():
    baseline_common_idx = baseline_common_idx.intersection(returns_series.index)
baseline_common_idx = baseline_common_idx.sort_values()

baseline_rows = []
for label, returns_series in baseline_return_candidates.items():
    returns_common = returns_series.reindex(baseline_common_idx).dropna()
    exposure_common = baseline_exposure_candidates[label].reindex(returns_common.index).ffill()
    metrics = calculate_performance_metrics(
        returns_common,
        loss_probability_horizons=LOSS_PROBABILITY_HORIZONS,
        loss_probability_bootstraps=LOSS_PROBABILITY_BOOTSTRAPS,
    )
    baseline_rows.append(
        {
            "strategy": label,
            "mean_exposure": exposure_common.mean(),
            **metrics,
        }
    )

baseline_comparison = pd.DataFrame(baseline_rows).set_index("strategy")
print(
    f"Baseline comparison window: {baseline_common_idx[0].date()} .. "
    f"{baseline_common_idx[-1].date()} ({len(baseline_common_idx)} days)"
)

baseline_display = baseline_comparison.copy()
for col in [
    "annualized_return",
    "max_drawdown",
    "cvar_95",
    "cvar_99",
    "loss_probability_252d",
    "annualized_volatility",
]:
    if col in baseline_display.columns:
        baseline_display[col] = baseline_display[col].map(lambda value: f"{value:.2%}")
for col in ["sortino_ratio", "calmar_ratio", "sharpe_ratio", "mean_exposure"]:
    if col in baseline_display.columns:
        baseline_display[col] = baseline_display[col].map(lambda value: f"{value:.3f}")
if "time_underwater_days" in baseline_display.columns:
    baseline_display["time_underwater_days"] = baseline_display["time_underwater_days"].map(
        lambda value: f"{int(value)}"
    )

baseline_metric_cols = [
    "mean_exposure",
    "annualized_return",
    "sortino_ratio",
    "max_drawdown",
    "calmar_ratio",
    "cvar_95",
    "time_underwater_days",
    "loss_probability_252d",
    "annualized_volatility",
    "sharpe_ratio",
]
baseline_display[[c for c in baseline_metric_cols if c in baseline_display.columns]]

In [ ]:
baseline_downside_checks = {
    "sortino_ratio": "higher",
    "max_drawdown": "higher",
    "calmar_ratio": "higher",
    "cvar_95": "lower",
}

hmm_scores = {}
for competitor in ["vix_threshold", "ma200"]:
    wins = 0
    for metric, direction in baseline_downside_checks.items():
        hmm_value = baseline_comparison.loc["hmm_exposure", metric]
        competitor_value = baseline_comparison.loc[competitor, metric]
        if direction == "higher" and hmm_value > competitor_value:
            wins += 1
        elif direction == "lower" and hmm_value < competitor_value:
            wins += 1
    hmm_scores[competitor] = wins

beats_both = all(score >= 3 for score in hmm_scores.values())
if beats_both:
    baseline_conclusion = (
        "The HMM overlay beats both simple baselines on at least three of the four "
        "downside checks, so it is adding value beyond trivial regime rules in this run."
    )
elif any(score >= 3 for score in hmm_scores.values()):
    winners = [name for name, score in hmm_scores.items() if score >= 3]
    baseline_conclusion = (
        "The HMM overlay beats only one simple baseline on the downside checks "
        f"({', '.join(winners)}). Treat the extra model complexity as only partly justified."
    )
else:
    baseline_conclusion = (
        "The HMM overlay does not beat the simple baselines on enough downside checks. "
        "A transparent VIX or moving-average rule may be preferable unless later tests improve."
    )

score_table = pd.Series(hmm_scores, name="hmm_wins_out_of_4").to_frame()
display(score_table)
display(Markdown(f"**Baseline interpretation:** {baseline_conclusion}"))

## 8. Sensitivity: Does The Result Depend On One HMM Setting?

The main run uses three states and a 5-year training window. Both choices are reasonable, but neither is sacred. The point of this section is to ask: if you nudge the model setup, does the conclusion survive?

The lightweight checks here are:

- **2 states instead of 3**: removes the `neutral` regime so the model has to choose between `risk_on` and `risk_off` only.
- **3-year training window instead of 5**: adapts faster to recent regime structure but is noisier and more sample-dependent.

This is intentionally not a full parameter search. It is a sanity check on stability. If the overlay conclusion flips entirely across these nearby variants, the result is parameter-sensitive and should be treated as a research finding, not a candidate production rule.

In [ ]:
sensitivity_specs = [
    {"label": "baseline_3_state_5y", "n_states": 3, "train_window": 252 * 5, "probs": regime_probs},
    {"label": "two_state_5y", "n_states": 2, "train_window": 252 * 5, "probs": None},
    {"label": "three_state_3y", "n_states": 3, "train_window": 252 * 3, "probs": None},
]

sensitivity_rows = []
for spec in sensitivity_specs:
    variant_probs = spec["probs"]
    if variant_probs is None:
        variant_probs = fit_regime_hmm(
            features,
            n_states=spec["n_states"],
            train_window=spec["train_window"],
            step=21,
            market_returns=mkt_returns,
        )

    variant_exposure = regime_signal(variant_probs, mode="exposure_scale").shift(1)
    variant_idx = unscaled.index.intersection(variant_exposure.dropna().index)
    variant_unscaled = unscaled.loc[variant_idx]
    variant_scaled = variant_unscaled * variant_exposure.reindex(variant_idx).ffill().fillna(0.5)
    variant_unscaled_metrics = calculate_performance_metrics(variant_unscaled)
    variant_scaled_metrics = calculate_performance_metrics(variant_scaled)
    variant_prob_cols = [col for col in variant_probs.columns if col.startswith("p_")]

    sensitivity_rows.append(
        {
            "variant": spec["label"],
            "n_states": spec["n_states"],
            "train_window_years": spec["train_window"] / 252,
            "days": len(variant_idx),
            "mean_exposure": variant_exposure.reindex(variant_idx).mean(),
            "avg_max_probability": variant_probs[variant_prob_cols].max(axis=1).mean(),
            "sharpe_delta": variant_scaled_metrics["sharpe_ratio"] - variant_unscaled_metrics["sharpe_ratio"],
            "calmar_delta": variant_scaled_metrics["calmar_ratio"] - variant_unscaled_metrics["calmar_ratio"],
            "max_drawdown_delta": variant_scaled_metrics["max_drawdown"] - variant_unscaled_metrics["max_drawdown"],
            "volatility_delta": variant_scaled_metrics["annualized_volatility"] - variant_unscaled_metrics["annualized_volatility"],
        }
    )

sensitivity_results = pd.DataFrame(sensitivity_rows).set_index("variant")
sensitivity_results.round(4)

### Sensitivity Interpretation

The deltas in the table are `regime_scaled - unscaled` for each variant. The sign convention to keep in mind:

- `sharpe_delta`, `calmar_delta`, `max_drawdown_delta`: **positive** is good.
- `volatility_delta`: **negative** means the overlay reduced volatility, which is descriptive but not by itself a verdict.

A robust result does not require every variant to match exactly. It does require the signs to point the same way for the metrics that decide downside risk. If only the baseline configuration works while the 2-state and 3-year variants disagree, the conclusion is parameter-sensitive and should be reported as a research idea, not a production-ready overlay.

In [ ]:
cum_unscaled = (1 + unscaled_common).cumprod()
cum_scaled = (1 + scaled).cumprod()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                                gridspec_kw={"height_ratios": [2, 1]})

ax1.plot(cum_unscaled.index, cum_unscaled.values, label="Unscaled Momentum",
         linewidth=1.0, color="#3498db")
ax1.plot(cum_scaled.index, cum_scaled.values, label="Regime-Scaled Momentum",
         linewidth=1.0, color="#e74c3c")
ax1.set_ylabel("Cumulative Return")
ax1.set_title("Equity Curves: Regime-Scaled vs Unscaled Momentum")
ax1.legend(loc="upper left")
ax1.grid(alpha=0.3)

dd_unscaled = calculate_drawdown(unscaled_common)
dd_scaled = calculate_drawdown(scaled)
ax2.fill_between(dd_unscaled.index, dd_unscaled.values, 0,
                 alpha=0.3, color="#3498db", label="Unscaled DD")
ax2.fill_between(dd_scaled.index, dd_scaled.values, 0,
                 alpha=0.3, color="#e74c3c", label="Regime-Scaled DD")
ax2.set_ylabel("Drawdown")
ax2.set_title("Drawdown Comparison")
ax2.legend(loc="lower left")
ax2.grid(alpha=0.3)

fig.tight_layout()
plt.show()

## 9. Drawdown Recovery Audit

The metrics table in section 6 can show a tradeoff that is easy to misread: the regime-scaled return can have a smaller worst drawdown but spend longer below its prior peak. Both can be true at the same time.

This section makes the tradeoff visible instead of leaving it inside aggregate numbers. It identifies the longest underwater stretch for the unscaled and HMM-scaled `mom_12_1` strategies, then plots the wealth curves around that stretch. The goal is diagnostic, not scoring.

There are two readings of a "shallower but longer underwater" result:

- **Acceptable:** the overlay reduced exposure during the drawdown so the loss was smaller, but it also kept exposure low into the recovery so the rebound was slower. That is a normal cost of scaling down.
- **Concerning:** the overlay kept exiting before positive-return periods, suppressing upside repeatedly. That is the overlay being too defensive rather than being well-calibrated.

Use the chart and the audit table to decide which reading applies to this run.

In [ ]:
def longest_underwater_summary(returns: pd.Series, label: str) -> dict:
    """Summarize the longest below-peak stretch for a return series."""
    clean_returns = returns.dropna()
    wealth = (1 + clean_returns).cumprod()
    running_peak = wealth.cummax()
    drawdown = (wealth - running_peak) / running_peak
    underwater = wealth < running_peak

    current_peak_date = None
    peak_dates = []
    current_peak_value = -np.inf
    for date, value in wealth.items():
        if value >= current_peak_value:
            current_peak_value = value
            current_peak_date = date
        peak_dates.append(current_peak_date)
    peak_date_by_date = pd.Series(peak_dates, index=wealth.index)

    longest = None
    start_pos = None
    for pos, is_underwater in enumerate(underwater.to_numpy()):
        if is_underwater and start_pos is None:
            start_pos = pos
        if (not is_underwater or pos == len(underwater) - 1) and start_pos is not None:
            end_pos = pos - 1 if not is_underwater else pos
            segment_index = wealth.index[start_pos : end_pos + 1]
            if longest is None or len(segment_index) > longest["underwater_days"]:
                trough_date = drawdown.loc[segment_index].idxmin()
                recovery_pos = end_pos + 1
                recovery_date = (
                    wealth.index[recovery_pos] if recovery_pos < len(wealth.index) else pd.NaT
                )
                peak_date = peak_date_by_date.loc[segment_index[0]]
                longest = {
                    "strategy": label,
                    "peak_date": peak_date,
                    "underwater_start": segment_index[0],
                    "trough_date": trough_date,
                    "recovery_date": recovery_date,
                    "underwater_end": segment_index[-1],
                    "underwater_days": len(segment_index),
                    "peak_to_trough_days": wealth.index.get_loc(trough_date) - wealth.index.get_loc(peak_date),
                    "trough_to_recovery_days": (
                        wealth.index.get_loc(recovery_date) - wealth.index.get_loc(trough_date)
                        if pd.notna(recovery_date)
                        else pd.NA
                    ),
                    "max_drawdown": drawdown.loc[segment_index].min(),
                }
            start_pos = None

    if longest is None:
        return {
            "strategy": label,
            "peak_date": pd.NaT,
            "underwater_start": pd.NaT,
            "trough_date": pd.NaT,
            "recovery_date": pd.NaT,
            "underwater_end": pd.NaT,
            "underwater_days": 0,
            "peak_to_trough_days": 0,
            "trough_to_recovery_days": 0,
            "max_drawdown": 0.0,
        }
    return longest


underwater_audit = pd.DataFrame(
    [
        longest_underwater_summary(unscaled_common, "unscaled_mom"),
        longest_underwater_summary(scaled, "hmm_scaled_mom"),
    ]
).set_index("strategy")

underwater_display = underwater_audit.copy()
for col in ["peak_date", "underwater_start", "trough_date", "recovery_date", "underwater_end"]:
    underwater_display[col] = underwater_display[col].map(
        lambda value: "still underwater" if pd.isna(value) else value.date()
    )
underwater_display["max_drawdown"] = underwater_display["max_drawdown"].map(
    lambda value: f"{value:.2%}"
)
underwater_display

In [ ]:
audit_start = min(underwater_audit["peak_date"].dropna()) - pd.offsets.BDay(63)
audit_end_candidates = underwater_audit["recovery_date"].dropna()
if audit_end_candidates.empty:
    audit_end = common_idx[-1]
else:
    audit_end = max(audit_end_candidates) + pd.offsets.BDay(63)

audit_window = common_idx[(common_idx >= audit_start) & (common_idx <= audit_end)]
wealth_unscaled = (1 + unscaled_common).cumprod()
wealth_scaled = (1 + scaled).cumprod()

fig, (ax1, ax2) = plt.subplots(
    2,
    1,
    figsize=(14, 8),
    sharex=True,
    gridspec_kw={"height_ratios": [2, 1]},
)

ax1.plot(
    audit_window,
    wealth_unscaled.reindex(audit_window),
    label="Unscaled Momentum",
    color="#3498db",
)
ax1.plot(
    audit_window,
    wealth_scaled.reindex(audit_window),
    label="HMM-Scaled Momentum",
    color="#e74c3c",
)
ax1.set_title("Wealth Curves During Longest Underwater Windows")
ax1.set_ylabel("Wealth Index")
ax1.legend(loc="upper left")
ax1.grid(alpha=0.3)

for strategy, color in [("unscaled_mom", "#3498db"), ("hmm_scaled_mom", "#e74c3c")]:
    row = underwater_audit.loc[strategy]
    ax1.axvline(row["peak_date"], color=color, linestyle="--", alpha=0.5)
    ax1.axvline(row["trough_date"], color=color, linestyle=":", alpha=0.7)
    if pd.notna(row["recovery_date"]):
        ax1.axvline(row["recovery_date"], color=color, linestyle="-.", alpha=0.5)

ax2.fill_between(
    audit_window,
    calculate_drawdown(unscaled_common).reindex(audit_window),
    0,
    alpha=0.25,
    color="#3498db",
    label="Unscaled DD",
)
ax2.fill_between(
    audit_window,
    calculate_drawdown(scaled).reindex(audit_window),
    0,
    alpha=0.25,
    color="#e74c3c",
    label="HMM-Scaled DD",
)
ax2.set_title("Drawdown Depth During Audit Window")
ax2.set_ylabel("Drawdown")
ax2.legend(loc="lower left")
ax2.grid(alpha=0.3)

fig.tight_layout()
plt.show()

### Drawdown Recovery Interpretation

The audit table answers two questions explicitly:

- **How deep was each strategy's worst drawdown?** Smaller magnitude is better.
- **How long did each strategy stay below its prior peak?** Shorter is better.

If the scaled strategy is shallower but longer underwater, look at the wealth-curve chart to see why. If the scaled curve is flat during a market rebound, the overlay is participating less because `p_risk_on` stayed low into the recovery; that is a scaling artifact. If the scaled curve repeatedly steps in and out across small rebounds, the overlay may be flipping too often, which is a different problem and a sign to review HMM stability.

## 10. Where Does The Overlay Help Or Hurt?

Aggregate metrics can hide where the overlay actually pays off. This section breaks daily momentum returns down by regime label so we can see if the overlay is doing its job in the days where it is supposed to do its job.

The comparison is simple: average daily `mom_12_1` return in `risk_off` days, in `risk_on` days, and across all days, with and without the overlay.

A few things to keep in mind when reading the table:

- `risk_off` is not a synonym for "bad days for momentum." It is the HMM's label for days that look like a stressed market environment based on the feature set in section 2.
- Momentum can crash on sharp rebounds, not only on selloffs, so the worst days for `mom_12_1` will not all live inside `risk_off`.
- The success criterion is not that the overlay improves every `risk_off` day. It is that the overlay reduces the worst portfolio damage and improves the full-period downside-adjusted profile while staying out of the way during normal markets.

In [ ]:
risk_off_mask = regime_probs["regime"].reindex(common_idx).ffill() == "risk_off"
risk_on_mask = regime_probs["regime"].reindex(common_idx).ffill() == "risk_on"

regime_stats = pd.DataFrame({
    "regime": ["risk_off", "risk_on", "all"],
    "days": [
        risk_off_mask.sum(),
        risk_on_mask.sum(),
        len(common_idx),
    ],
    "unscaled_mean_return_bps": [
        unscaled_common[risk_off_mask].mean() * 1e4 if risk_off_mask.any() else 0,
        unscaled_common[risk_on_mask].mean() * 1e4 if risk_on_mask.any() else 0,
        unscaled_common.mean() * 1e4,
    ],
    "scaled_mean_return_bps": [
        scaled[risk_off_mask].mean() * 1e4 if risk_off_mask.any() else 0,
        scaled[risk_on_mask].mean() * 1e4 if risk_on_mask.any() else 0,
        scaled.mean() * 1e4,
    ],
})
regime_stats["reduction_bps"] = regime_stats["unscaled_mean_return_bps"] - regime_stats["scaled_mean_return_bps"]
regime_stats.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(exposure.index, exposure.values, linewidth=0.7, color="steelblue", label="Regime Exposure")
ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.5, alpha=0.5)

risk_off_dates = regime_probs.index[regime_probs["regime"] == "risk_off"]
for d in risk_off_dates:
    ax.axvline(d, color="red", alpha=0.02, linewidth=0.5)

ax.set_ylabel("Exposure Scale [0, 1]")
ax.set_title("Regime Exposure Signal (red = risk-off days)")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## Summary

This notebook asks one specific question:

> Can we infer market risk regimes from observable market, factor, macro, and volatility data, and can those regimes reduce downside risk when used as an exposure overlay on a momentum strategy?

The chain of logic, in plain English:

1. Build a feature matrix where every column is documented: source, transformation, missing-data handling, publication lag, when it gets standardized, and how it could leak.
2. Sanity-check those features around known stress periods so we are not asking the HMM to discover structure that is not visibly there.
3. Fit a walk-forward 3-state HMM, then name the states using trailing market-return behavior so labels like `risk_on` and `risk_off` mean something specific.
4. Convert filtered `p_risk_on` into an exposure scale from 0% to 100%, lag it by one day, and apply it as a multiplier on top of an unchanged `mom_12_1` long/short return.
5. Compare the unscaled and scaled momentum strategies using downside-risk metrics first, with volatility and Sharpe kept only as context.
6. Repeat the same overlay rule on a small set of nearby price-based factors to see if the overlay helps broadly or only for `mom_12_1`.
7. Compare the HMM overlay against two transparent baselines: a VIX threshold and a 200-day moving-average filter. The HMM only earns its complexity if it clearly beats those.
8. Stress the configuration with a 2-state variant and a 3-year training window to check that the conclusion is not driven by one HMM setting.
9. Audit the longest underwater stretch directly, so a shallower-but-longer drawdown can be diagnosed instead of just shown.
10. Inspect where the overlay actually helps or hurts inside `risk_off` and `risk_on` days.

### How To Read The Generated Conclusions

The conclusions printed under sections 6, 7, and 8 are generated from the current run's metric deltas. Read them together with the delta tables, not in isolation. They say, for this exact data and configuration, whether the overlay improved downside risk and whether the HMM added value beyond the simple baselines.

If the answer is yes across the downside checks, breadth check, baselines, and sensitivity variants, the overlay is a candidate for further work. If any of those checks fails, the signal is a research idea, not a production overlay.

### What Counts As Evidence

- **For the overlay**: lower max drawdown, lower CVaR(95) and CVaR(99), lower 1-year loss probability, higher Sortino and Calmar, with at least directional agreement across nearby factors and HMM variants, and clear improvement over the VIX-threshold and 200-day baselines.
- **Against the overlay**: lower volatility but no improvement in downside-risk metrics; only `mom_12_1` benefits; the simple baselines do as well or better; the conclusion flips under 2-state or 3-year variants; or the overlay reduces drawdown depth only by sitting out long stretches of recovery.

### Disclosed Risks And Limitations

- Macro publication lags are conservative approximations, not true ALFRED point-in-time vintage data.
- Long/short returns are gross of borrow cost, short availability, and merger-payout recovery. Delisted names are realized at -100%.
- The HMM starts producing signals only after the first full training window, so the backtest starts around 2004-11.
- Three states were chosen as an interpretable starting point, not because a held-out likelihood search proved three is optimal.
- HMM parameters and label mapping are held fixed inside each 21-day prediction block. Filtered probabilities still update daily inside the block.
- HMM probabilities near 0 or 1 can mean clean separation or unstable fits. The confidence tables in section 4 are the place to check that.
- The current factor set is limited to price-derived factors. Value, quality, and profitability factors require additional data before they can be tested here.

### Next Steps

1. Replace conservative FRED lags with true ALFRED/FRED vintage data where available.
2. Expand the sensitivity grid to include 4-state HMMs and 10-year training windows.
3. Add an explicit out-of-sample slice for parameter selection instead of judging all variants on the same period.
4. Add borrow-cost, short-availability, and delisting-recovery assumptions so long/short results are realistic, not idealized.
5. If the overlay survives baselines, breadth, and out-of-sample testing, decide whether to expose it as a reusable regime signal in `core/signals/` and the strategy/API layers.